# WB3 — 데이터 기반 카드뉴스 (student)

CPU, API 키 불필요. 독립 실습. 행사 정보와 신청자 수는 모두 교육용 가상 데이터입니다. 사람 초안을 먼저 작성한 뒤 AI에 분석 결과만 전달하여 4장 카드뉴스를 완성합니다.

## 코드 셀 1. 데이터 읽기

**코드 설명:** read_csv는 로컬→GitHub→내장 순으로 읽습니다. 행사 표의 id를 확인합니다.

**체크포인트:** 행사 4행

In [ ]:
import pandas as pd, io, json
from pathlib import Path
from urllib.request import urlopen
from IPython.display import display, HTML
def read_csv(name, fallback):
    p = Path(name)
    if p.exists():
        raw = p.read_text(encoding="utf-8-sig")
    else:
        try:
            raw = urlopen("https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/week13/"+name, timeout=6).read().decode("utf-8-sig")
        except Exception:
            raw = fallback
    return pd.read_csv(io.StringIO(raw))
events = read_csv("event_briefs.csv", 'id,event,date,time,place,capacity,fee,deadline,audience,action\nA01,캠퍼스 북교환회,2026-12-03,14:00~16:00,학생회관 1층,40,무료,2026-12-01,재학생,도서 1권 지참 후 신청\nA02,디지털 사진 전시회,2026-12-04,13:00~17:00,도서관 로비,60,무료,2026-12-02,재학생,온라인 신청\nA03,텀블러 데이,2026-12-07,12:00~14:00,학생식당 앞,80,무료,2026-12-05,재학생,개인 텀블러 지참\nA04,진로 포트폴리오 특강,2026-12-08,15:00~17:00,강의동 201호,30,무료,2026-12-06,재학생,온라인 신청\n')
assert len(events) == 4 and events["id"].is_unique
display(events)


## 코드 셀 2. 신청 데이터 결합·계산

**코드 설명:** merge는 id 기준으로 두 표를 결합합니다. validate는 일대일 관계를 점검합니다. remaining은 정원-신청자, rate는 신청자/정원×100입니다. sum은 전체 합, idxmax는 가장 큰 신청률의 행입니다.

**체크포인트:** 정원 210, 신청 160, 잔여 50, 전체 76.2%, 최고 A04 90%

In [ ]:
reg = read_csv("registration_demo.csv", 'id,registered\nA01,32\nA02,45\nA03,56\nA04,27\n')
assert reg["id"].is_unique and len(reg)==4
df = events.merge(reg,on="id",validate="one_to_one")
assert len(df)==4 and (df["registered"]<=df["capacity"]).all()
df["remaining"] = df["capacity"] - df["registered"]
df["rate"] = (df["registered"]/df["capacity"]*100).round(1)
display(df[["id","event","capacity","registered","remaining","rate"]])
summary = {"total_capacity":int(df.capacity.sum()),"total_registered":int(df.registered.sum()),"total_remaining":int(df.remaining.sum()),"highest_rate_event":str(df.loc[df.rate.idxmax(),"event"]),"highest_rate":float(df.rate.max())}
summary["overall_rate"] = round(summary["total_registered"]/summary["total_capacity"]*100,1)
print(json.dumps(summary,ensure_ascii=False,indent=2))


## 코드 셀 3. 그래프 생성

**코드 설명:** set_index는 행사 ID를 축으로 사용합니다. plot.bar는 두 수치를 막대로 비교하고 savefig는 PNG로 저장합니다.

**체크포인트:** 막대 4쌍, wb3_chart.png 생성

In [ ]:
import matplotlib.pyplot as plt
ax = df.set_index("id")[["capacity","registered"]].plot.bar(color=["#94a3b8","#0d9488"])
ax.set_ylabel("People"); ax.set_xlabel("Event ID"); ax.set_title("Capacity vs registrations")
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig("wb3_chart.png",dpi=140)
plt.show()


## 코드 셀 4. AI 프롬프트

**코드 설명:** json.dumps와 to_json은 계산한 수치를 구조화합니다. f-string은 데이터와 조건을 문장에 넣습니다. AI 웹에 붙여넣기 전 사람이 직접 4장 문구를 먼저 작성합니다.

**체크포인트:** 카드뉴스 프롬프트 출력

In [ ]:
prompt = f"""역할: 대학 학생회 카드뉴스 편집자.
다음 교육용 가상 행사 신청 현황만 사용해 4개 카드 제목/본문을 작성.
1: 전체 현황, 2: 신청률, 3: 행사별 특징, 4: 참여 안내.
근거: {json.dumps(summary,ensure_ascii=False)}
행사별 수치: {df[["id","event","capacity","registered","remaining","rate"]].to_json(orient="records",force_ascii=False)}
추가 사실 금지, 신청률과 방문율 혼동 금지, 카드당 80자 이내."""
print(prompt)


## 코드 셀 5. 카드뉴스 생성

**코드 설명:** cards는 제목/본문 쌍의 리스트입니다. enumerate는 번호를 부여하고 escape는 입력을 안전한 HTML 텍스트로 변환합니다. base64는 그래프를 HTML에 포함합니다. 학생은 직접 검증한 AI 문구로 빈칸을 수정합니다.

**체크포인트:** wb3_cards.html 생성; 카드 4개

In [ ]:
from html import escape
import base64
cards = [('캠퍼스 행사 신청 현황', ''), ('전체 신청률', ''), ('행사별 특징', ''), ('참여 안내', '')]
# 학생은 빈 문구를 실제 AI 응답을 확인·수정한 뒤 채웁니다.
chart_b64 = base64.b64encode(Path("wb3_chart.png").read_bytes()).decode()
body = ""
for i, (title, text) in enumerate(cards, 1):
    image = f'<img src="data:image/png;base64,{chart_b64}" alt="행사별 정원과 신청자 그래프" style="width:100%">' if i==3 else ""
    body += f'<section><small>가상 데이터 · {i}/4</small><h1>{escape(title)}</h1><p>{escape(text or "문구를 작성하세요")}</p>{image}</section>'
html = '<!doctype html><html lang="ko"><meta charset="utf-8"><title>가상 행사 카드뉴스</title><style>body{font-family:sans-serif;background:#e2e8f0;display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:16px;padding:16px}section{background:white;border-radius:16px;padding:30px;min-height:340px;border-top:9px solid #0d9488}h1{color:#124b45}p{font-size:22px;line-height:1.6}small{color:#475569}</style>'+body+'</html>'
Path("wb3_cards.html").write_text(html,encoding="utf-8")
display(HTML(html))
print("저장: wb3_cards.html")


## 코드 셀 6. 기록 및 제출

**코드 설명:** 사람 초안과 AI 비교 결과를 기록하고 JSON으로 저장합니다. 실제 사용 도구와 날짜, 사실 검증을 기록합니다.

**체크포인트:** wb3_record.json, HTML, PNG 다운로드

In [ ]:
record = {'tool_and_date': '', 'human_draft': '', 'basic_result_observation': '', 'improved_result_observation': '', 'fact_review': ''}
Path("wb3_record.json").write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding="utf-8")
print("저장: wb3_record.json — 학생은 기록 칸을 채우세요.")
